# 教師の確認（life #24・2026-11-02 の協議と本人の承認・Colab L4 用）

「賢いモデルを小さくする」研究の蒸留の入口。**Qwen3.8-27B の 5bit（unsloth UD-Q5_K_M）を教師として、今までの評価で使っていない 6 規程・96 問に答えさせる。** 速さは問わない。

**測る前に固定した事**（life #24 のコメントと同じ。測った後に変えない）:

- 重み: `unsloth/Qwen3.8-27B-GGUF` の `Qwen3.8-27B-UD-Q5_K_M.gguf`（revision `4ca7207`）
- 実行系: llama.cpp b11836 の配布版 llama-server（`-ngl 999 --jinja --reasoning off -np 1 -c 4096`）。GPU は L4（24GB）
- 思考 OFF・温度 0.1・出力の上限 1,024 トークン
- 検索は使わず、問の規程の全文をそのまま渡す（`evaluation/teacher_check/build.py` の `prompt_messages`）
- 問題集: `evaluation/teacher_check/sealed/questions.jsonl`（96 問・境目 48・言い換え 24・例外と保留 24）。sha256 が封の値と違えば止める
- 採点: 最後の「結論:」行が正解と完全一致し、かつ最後の「根拠:」行に正解の条が 1 つ以上ある（`score.py`・コードで照合）

**合格と中止**:

- 別々に起動した 2 回とも 96/96 で合格。**1 問でも誤れば、その時点でこの教師は中止**（答えられる問を「判断できない」とした時も誤り）
- 順番: 1 回目 = 先に決めた 12 問（区分ごとに 4 問）→ 残り 84 問、2 回目 = 起動し直して 96 問
- 失敗した問を見て設定や問を変えない。準備の失敗（GPU が取れない・起動しない・通信の失敗）だけは、その回を丸ごとやり直してよい
- 合格しても、実用の合格や F7 の達成とは扱わない。蒸留した後の生徒は §1 の全部の線を 2 回とも通す必要がある

使い方: ランタイムを **L4 GPU** にして「すべてのセルを実行」。

In [ ]:
!git clone -q --depth 1 -b research/teacher-check https://github.com/Maaa2005/local-rag-agent.git /content/repo
!git -C /content/repo log -1 --format='repo HEAD %H %s'

In [ ]:
# 設定（測る前に固定・測った後に変えない）
MODEL = "Qwen/Qwen3.8-27B"
GGUF_REPO = "unsloth/Qwen3.8-27B-GGUF"
GGUF_REV = "4ca720788d1e01f1bff70c033e0d0028fd02e502"
GGUF_FILE = "Qwen3.8-27B-UD-Q5_K_M.gguf"
LLAMA_TAG = "b11836"
LLAMA_ASSET = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_ASSET = f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
CTX = 4096
MAX_TOKENS = 1024
TEMPERATURE = 0.1
QUESTIONS_SHA256 = "5f1bf9a17f413de6ada834961fd27a8363b5461cb9d7366cf8601b3f62a5371f"            # 封をした問題集の sha256（life #24 に書いた値）
REPO = "/content/repo"
OUT = "/content/teacher"
PORT = 8001
REQUEST_TIMEOUT = 600

In [ ]:
import base64, gzip, hashlib, json, os, pathlib, re, signal, subprocess, sys, time, urllib.request

sys.path.insert(0, REPO)
from evaluation.teacher_check import build as tb
from evaluation.teacher_check import score as ts

OUTP = pathlib.Path(OUT)
if OUTP.exists() and any(OUTP.iterdir()):
    raise RuntimeError(f"{OUT} が空でない（前の結果と混ざるので止める）")
OUTP.mkdir(parents=True, exist_ok=True)

STAGES = []
STOP = None
def stage(name, result, reason=None, **nums):
    global STOP
    rec = {"stage": name, "result": result, "reason": reason, **nums}
    STAGES.append(rec)
    if result in ("fail", "prep_failure") and STOP is None:
        STOP = rec
    print("STAGE_JSON " + json.dumps(rec, ensure_ascii=False))

def prep_fail(name, reason):
    stage(name, "prep_failure", reason)
    raise RuntimeError(f"準備の失敗（{name}）: {reason}")

GPU_NAME, TOTAL_MIB = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.splitlines()[0].split(",")]
if "L4" not in GPU_NAME:
    prep_fail("GPU", f"L4 でない: {GPU_NAME}")
REPO_SHA = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

# 問題集の封を確かめる
QPATH = pathlib.Path(REPO) / "evaluation/teacher_check/sealed/questions.jsonl"
qsha = hashlib.sha256(QPATH.read_bytes()).hexdigest()
if qsha != QUESTIONS_SHA256:
    prep_fail("封", f"問題集の sha256 が違う: {qsha}")
for name, h in json.loads((QPATH.parent / "manifest.json").read_text())["corpus_sha256"].items():
    if hashlib.sha256((tb.CORPUS / name).read_bytes()).hexdigest() != h:
        prep_fail("封", f"規程 {name} の sha256 が違う")
ROWS = [json.loads(l) for l in QPATH.read_text(encoding="utf-8").splitlines() if l.strip()]
STAGE1 = [r for r in ROWS if r["stage1"]]
REST = [r for r in ROWS if not r["stage1"]]
assert len(ROWS) == 96 and len(STAGE1) == 12

# llama-server（配布版・ubuntu・CUDA 12.8）
pre = pathlib.Path("/content/llama-pre"); pre.mkdir(exist_ok=True)
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
for a in (LLAMA_ASSET, CUDART_ASSET):
    r = subprocess.run(["bash", "-c", f"set -o pipefail; curl -sfL {base}/{a} | tar xz -C {pre}"], capture_output=True, text=True)
    if r.returncode != 0:
        prep_fail("llama-server", f"{a} が取れない: {r.stderr[-300:]}")
SERVER_BIN = str(sorted(pre.rglob("llama-server"))[0])
libdirs = sorted({str(p.parent) for p in pre.rglob("*.so*")})
LLAMA_ENV = dict(os.environ, LD_LIBRARY_PATH=":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
print("GPU", GPU_NAME, TOTAL_MIB, "MiB / repo", REPO_SHA, "/ 問題集", qsha)
stage("準備（GPU・封・実行系）", "pass", gpu=GPU_NAME, repo=REPO_SHA, questions_sha256=qsha)

In [ ]:
# 重み（revision 固定・トークン不要）。進み具合の表示があるので、このセルだけ別にする
from huggingface_hub import hf_hub_download
t = time.time()
try:
    GGUF_PATH = hf_hub_download(GGUF_REPO, GGUF_FILE, revision=GGUF_REV, local_dir="/content/gguf")
except Exception as e:
    prep_fail("重み", f"取れない: {repr(e)[:300]}")
GGUF_GIB = round(os.path.getsize(GGUF_PATH) / 2**30, 2)

In [ ]:
print("重み", GGUF_FILE, GGUF_GIB, "GiB")
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

def launch(argv, log, env=None):
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def kill(proc):
    if proc is None:
        return
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait()

def start_server(label):
    log = OUTP / f"llama_{label}.log"
    argv = [SERVER_BIN, "-m", GGUF_PATH, "-ngl", "999", "-c", str(CTX), "--jinja", "--reasoning", "off",
            "-np", "1", "--alias", "teacher", "--host", "127.0.0.1", "--port", str(PORT)]
    proc = launch(argv, log, LLAMA_ENV)
    t = time.time()
    while time.time() - t < 900:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3)
            text = log.read_text(errors="replace")
            off = re.findall(r"offloaded (\d+)/(\d+) layers to GPU", text)
            ok = bool(off) and off[-1][0] == off[-1][1]
            print(f"[{label}] 起動 {time.time() - t:.0f} 秒 / 層 {off[-1] if off else None}")
            return proc, ok
        except Exception:
            time.sleep(3)
    print("\n".join(log.read_text(errors="replace").splitlines()[-30:]))
    kill(proc)
    return None, False

def chat(messages, max_tokens=MAX_TOKENS):
    body = json.dumps({"model": "teacher", "messages": messages, "temperature": TEMPERATURE,
                       "max_tokens": max_tokens, "stream": False}).encode()
    req = urllib.request.Request(f"{BASE_URL}/chat/completions", body, {"Content-Type": "application/json"})
    t = time.time()
    res = json.loads(urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT).read())
    ch = res["choices"][0]
    msg = ch["message"]
    return {"content": msg.get("content") or "", "reasoning_chars": len(msg.get("reasoning_content") or ""),
            "finish_reason": ch.get("finish_reason"), "usage": res.get("usage"), "seconds": round(time.time() - t, 1)}

def precheck():
    r = chat([{"role": "user", "content": "日本の首都はどこですか。一文で答えてください。"}], max_tokens=64)
    ok = bool(r["content"]) and r["reasoning_chars"] == 0 and not re.search(r"<\|?think|</think>", r["content"])
    print("PRECHECK_JSON " + json.dumps({k: r[k] for k in ("reasoning_chars", "finish_reason", "usage")} | {"ok": ok}, ensure_ascii=False))
    return ok

ANSWERS = []   # 全回答（回・問・本文・採点）
def run_set(rows, label):
    # 1 問ずつ流し、誤り（結論か根拠の条が違う・上限で切れて結論が無い）が出たらその時点で止める。返り値は誤った問か None
    for r in rows:
        try:
            out = chat(tb.prompt_messages(r))
        except Exception as e:
            prep_fail(f"{label} の通信", f"{r['id']}: {repr(e)[:200]}")
        g = ts.grade(out["content"], r["expected_conclusion"], r["expected_articles"])
        rec = {"run": label, "id": r["id"], "category": r["category"], **g, "finish_reason": out["finish_reason"],
               "reasoning_chars": out["reasoning_chars"], "usage": out["usage"], "seconds": out["seconds"]}
        ANSWERS.append(dict(rec, answer=out["content"]))
        print("Q_JSON " + json.dumps(rec, ensure_ascii=False))
        if (out["usage"] or {}).get("prompt_tokens", 0) + MAX_TOKENS > CTX:
            prep_fail(f"{label} の文脈長", f"{r['id']} の入力が文脈長に収まらない")
        if out["reasoning_chars"]:
            prep_fail(f"{label} の思考 OFF", f"{r['id']} で考える文が出た")
        if not g["ok"]:
            return r["id"]
    return None

In [ ]:
# 1 回目: 先に決めた 12 問 → 残り 84 問
srv, ok = start_server("run1")
if not ok:
    prep_fail("1 回目の起動", "起動しない・全層が GPU に載らない")
if not precheck():
    kill(srv)
    prep_fail("1 回目の思考 OFF", "事前の 1 問で考える文が出た・本文が空")
bad = run_set(STAGE1, "run1-stage1")
if bad:
    stage("1 回目・先の 12 問", "fail", f"{bad} を誤った")
else:
    stage("1 回目・先の 12 問", "pass")
    bad = run_set(REST, "run1-rest")
    stage("1 回目・残り 84 問", "fail" if bad else "pass", f"{bad} を誤った" if bad else None)
kill(srv)

In [ ]:
# 2 回目: 起動し直して 96 問（1 回目を通った時だけ）
if STOP:
    stage("2 回目・96 問", "skipped", f"前の段で終了: {STOP['stage']}")
else:
    time.sleep(10)
    srv, ok = start_server("run2")
    if not ok:
        prep_fail("2 回目の起動", "起動しない・全層が GPU に載らない")
    if not precheck():
        kill(srv)
        prep_fail("2 回目の思考 OFF", "事前の 1 問で考える文が出た・本文が空")
    bad = run_set(ROWS, "run2")
    stage("2 回目・96 問", "fail" if bad else "pass", f"{bad} を誤った" if bad else None)
    kill(srv)

In [ ]:
# まとめ（数だけ）と、全回答の控え（gzip+base64。人が計算の中身を抜き取りで読むため）
by_run = {}
for a in ANSWERS:
    s = by_run.setdefault(a["run"], {"n": 0, "ok": 0})
    s["n"] += 1; s["ok"] += a["ok"]
SUMMARY = {"model": MODEL, "gguf": GGUF_FILE, "revision": GGUF_REV, "gib": GGUF_GIB, "runtime": f"llama.cpp {LLAMA_TAG}",
           "gpu": GPU_NAME, "repo": REPO_SHA, "questions_sha256": qsha, "ctx": CTX, "max_tokens": MAX_TOKENS,
           "temperature": TEMPERATURE, "by_run": by_run, "stages": STAGES, "stopped_at": STOP,
           "result": "fail" if STOP and STOP["result"] == "fail" else ("prep_failure" if STOP else "pass")}
print("SUMMARY_JSON " + json.dumps(SUMMARY, ensure_ascii=False))
(OUTP / "answers.jsonl").write_text("".join(json.dumps(a, ensure_ascii=False) + "\n" for a in ANSWERS), encoding="utf-8")
blob = base64.b64encode(gzip.compress((OUTP / "answers.jsonl").read_bytes())).decode()
print(f"ANSWERS_B64_LEN {len(blob)}")
print("ANSWERS_B64_BEGIN")
for i in range(0, len(blob), 10000):
    print(blob[i:i + 10000])
print("ANSWERS_B64_END")
for s in STAGES:
    print(f"{s['stage']}: {s['result']}" + (f"（{s['reason']}）" if s.get("reason") else ""))